# Inferensi Time-Series + Output Kontrak WebGIS (Tahap 2-3)

Notebook ini mengambil komposit citra (image-only, 36 tile/tahun) yang sudah di-export ke
Google Drive (lihat `export_inference_years.ipynb`), menjalankan model 7-kelas, lalu menyusun
**semua file yang dibutuhkan WebGIS Kasuari AI** ke SATU folder Drive baru: **`Output_Fix`**.

**Alur:**
1. Load model (checkpoint `.pt`) sekali.
2. Inferensi tile-by-tile per tahun yang tersedia (`papua_{tahun}_tile_*.tif` -> mask lokal).
3. Mosaic mask -> `landcover_{tahun}.png` + `landcover_{tahun}_bounds.json` per tahun
   (Sekarang dibuat untuk **semua tahun** yang ada, bukan cuma 2021/2025 -- bonus utk
   time-series slider yang sudah ada di frontend).
4. Change detection **kumulatif 2021->2025** (T1 vs T2) -> `deforestation.geojson`
   (kontrak WebGIS cuma punya 1 file ini, jadi tak dibuat per-tahun-pasangan).
5. `statistics.json`, `legend.json`, `metrics.json`, `model_card.md`.
6. Export ONNX dari model yang BARU SAJA dipakai inferensi (bukan copy file lama yg mungkin
   beda checkpoint) -> `model.onnx`.

**Catatan jujur -- gap data (`per_province` di statistics.json):**
Deteksi perubahan butuh nama provinsi per polygon untuk chart "Peringkat Provinsi" di
Dashboard. **Tidak ada dataset batas 6 provinsi Papua (pasca pemekaran 2022) di repo ini** --
sudah dicek: `dummy.py` cuma random-pilih provinsi, tak ada shapefile/GeoJSON batas asli.
Notebook ini punya slot opsional (`PROVINCE_BOUNDARY_GEOJSON`, Bagian 4) -- isi kalau sudah
punya file batas provinsi; kalau dikosongkan (default), polygon tetap dibuat tapi TANPA tag
provinsi (chart per-provinsi akan kosong utk data asli, tapi semua data lain tetap lengkap).


## Bagian 0 -- Setup environment (Colab) + mount Drive

Tak perlu auth GEE di notebook ini (cuma baca GeoTIFF tile dari Drive, tak ada panggilan GEE).


In [ ]:
# === Bagian 0 -- Setup environment (Colab) + mount Drive ===
import os, sys, subprocess
from pathlib import Path

subprocess.run("cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
               "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
               shell=True, check=False)
subprocess.run('pip install -q -e "/content/fw_repo/model[ml,gis]"', shell=True, check=False)

_SRC = "/content/fw_repo/model/src"
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

from google.colab import drive  # noqa: PLC0415
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
print("Drive ter-mount:", DRIVE_ROOT.exists())


## Bagian 1 -- Config

Set tahun yang mau diproses, lokasi tile di Drive, checkpoint model, dan folder output baru
(`Output_Fix`). Tahun yang foldernya belum ada di Drive otomatis dilewati (bukan error) --
jalankan ulang notebook ini kapan saja tahun baru selesai export.


In [ ]:
# === Bagian 1 -- Config ===
YEARS = [2021, 2022, 2023, 2024, 2025]              # tahun yg DICOBA diproses (skip kalau blm ada)
T1_YEAR, T2_YEAR = 2021, 2025                       # pasangan resmi utk deforestation.geojson

# Semua aset proyek (tile, training, checkpoint) hidup di SATU folder ini di Drive kamu --
# dicek langsung dari Drive: "Drive Saya > Satria Data 3.0 > ...".
PROJECT_ROOT = DRIVE_ROOT / "Satria Data 3.0"

TILE_DIR_TMPL = "ForestWatch_Tiles_{year}"          # folder DI BAWAH PROJECT_ROOT, image-only
TILE_GLOB_TMPL = "papua_{year}_tile_*.tif"

# 2021 (T1) & 2025 (T2) BUKAN folder per-tahun -- itu export awal (full pipeline + label),
# namanya tetap "ForestWatch_Tiles_T1"/"_T2" (dicek dari Drive). Naming file di dalamnya
# belum diverifikasi, jadi glob dibuat permisif ("*.tif") utk folder ini saja.
TILE_DIR_OVERRIDE = {T1_YEAR: "ForestWatch_Tiles_T1", T2_YEAR: "ForestWatch_Tiles_T2"}
TILE_GLOB_OVERRIDE = {T1_YEAR: "*.tif", T2_YEAR: "*.tif"}

OUT_DIR = PROJECT_ROOT / "Output_Fix"               # <- folder baru, SEMUA output webgis di sini
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Tile asli (6 band float32) BISA >9 GB/tile -- jangan pernah disimpan menumpuk di disk lokal
# Colab (~100 GB). Pola streaming (Bagian 3): copy 1 tile -> infer -> hapus segera.
LOCAL_TILE_STAGE_DIR = Path("/content/tile_stage")
LOCAL_TILE_STAGE_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint + metrik TEST -- dicek langsung dari Drive:
# Satria Data 3.0/Bahan_Training_Fix_Combined_v4/output/{best_model_finetune_v2.pt,metrics_finetune.json}
TRAINING_OUTPUT_DIR = PROJECT_ROOT / "Bahan_Training_Fix_Combined_v4" / "output"
MODEL_CHECKPOINT_PATH = TRAINING_OUTPUT_DIR / "best_model_finetune_v2.pt"
MODEL_ARCH = dict(architecture="unet_scse", encoder_name="resnet50", encoder_weights=None)

# Sesi sebelumnya OOM ("session crashed after using all available RAM") -- akumulator
# probabilitas overlap-blending (n_classes x H x W float32) ternyata MASIH terlalu besar
# utk Colab free (~12-13 GB), meski citra sumber sudah windowed-read (tak full-load).
# FIX: STRIDE = PATCH_SIZE (non-overlap) -- infer_tile() lalu otomatis pakai jalur HEMAT
# RAM (tulis argmax langsung ke mask uint8, TANPA akumulator float32 7-kelas). Trade-off:
# kehilangan blending-halus antar-window DI DALAM 1 tile (cuma di strip tepi kalau H/W
# tile bukan kelipatan 256px) -- bukan korup, cuma sedikit kurang mulus drpd overlap.
IN_CHANNELS, PATCH_SIZE, STRIDE, TTA = 6, 256, 256, False

# Jumlah sliding-window digabung per forward-pass GPU (bukan "worker"/multiprocessing --
# cuma ada 1 GPU, proses paralel antar-tile akan rebutan GPU yg sama, bukan mempercepat).
# Diturunkan ke 4 (dari 16) krn sesi sebelumnya OOM -- naikkan lagi kalau RAM ternyata longgar.
INFER_BATCH_SIZE = 4

# Cache mask hasil inferensi -- DISIMPAN DI DRIVE (bukan /content), supaya:
# (a) tile yg SUDAH diproses tak hilang kalau runtime Colab disconnect/restart di tengah jalan
#     (notebook ini aman dijalankan ulang kapan saja -- tile yg sudah ada mask-nya di-skip).
# (b) ganti checkpoint/patch_size/stride otomatis bikin cache baru (tag di nama folder),
#     jadi TAK PERNAH diam-diam mencampur mask dari config lama & baru.
CACHE_TAG = f"{MODEL_CHECKPOINT_PATH.stem}_p{PATCH_SIZE}_s{STRIDE}"
MASK_CACHE_ROOT = PROJECT_ROOT / "_InferenceCache" / CACHE_TAG

# Metrik TEST hasil training -- file ini SUDAH ADA (terlihat di Drive), jadi statistics.json/
# metrics.json akan pakai angka asli, bukan kosong/dikarang.
METRICS_SRC_JSON = TRAINING_OUTPUT_DIR / "metrics_finetune.json"

# Opsional: GeoJSON batas 6 provinsi Papua (utk tag `province` di tiap polygon perubahan).
# Kosongkan (None) kalau belum punya -- lihat catatan gap data di markdown atas.
PROVINCE_BOUNDARY_GEOJSON = None
PROVINCE_NAME_FIELD = "province"  # nama field di properties GeoJSON boundary yg berisi nama provinsi

MIN_AREA_HA = 0.5
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"

print("PROJECT_ROOT   :", PROJECT_ROOT, "| ada:", PROJECT_ROOT.exists())
print("OUT_DIR        :", OUT_DIR)
print("MASK_CACHE_ROOT:", MASK_CACHE_ROOT, "(tag config:", CACHE_TAG, ")")
print("MODEL_CHECKPOINT_PATH:", MODEL_CHECKPOINT_PATH, "| ada:", MODEL_CHECKPOINT_PATH.exists())
print("METRICS_SRC_JSON     :", METRICS_SRC_JSON, "| ada:", METRICS_SRC_JSON.exists())
print("DEVICE         :", DEVICE, "-- GANTI ke GPU (Runtime > Change runtime type) kalau masih 'cpu'.")
print("STRIDE         :", STRIDE, "(== PATCH_SIZE -> mode non-overlap, hemat RAM)")
print("INFER_BATCH_SIZE:", INFER_BATCH_SIZE)
print("Province join  :", "AKTIF" if PROVINCE_BOUNDARY_GEOJSON else "DILEWATI (gap data, lihat catatan)")


## Bagian 2 -- Load model (sekali, dipakai utk semua tahun + export ONNX)

In [ ]:
# === Bagian 2 -- Load model ===
import torch
from forestwatch.model.architecture import build_unet
from forestwatch.constants import N_CLASSES

assert MODEL_CHECKPOINT_PATH.exists(), f"Checkpoint tak ditemukan: {MODEL_CHECKPOINT_PATH}"

model = build_unet(in_channels=IN_CHANNELS, classes=N_CLASSES, **MODEL_ARCH).to(DEVICE)
model.load_state_dict(torch.load(MODEL_CHECKPOINT_PATH, map_location=DEVICE))
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model siap ({n_params:,} parameter) di {DEVICE}. Checkpoint: {MODEL_CHECKPOINT_PATH.name}")


## Bagian 3 -- Inferensi per tahun (STREAMING + PREFETCH AMAN + RETRY + RESUMABLE)

Tile asli (6 band float32) bisa **>9 GB/tile** -- jangan pernah dibaca langsung lewat Drive
FUSE mount satu-satu (lambat, banyak random-read) ATAU disalin semua ke disk lokal Colab
sekaligus (disk lokal cuma ~100 GB, total tile per tahun bisa ratusan GB).

**Riwayat:** sempat dicoba `PREFETCH_WORKERS=4` (4 download SEKALIGUS) supaya GPU yg
nganggur saat nunggu copy bisa terisi -- tapi ini bikin Drive FUSE mount Colab **disconnect**
(`OSError: Transport endpoint is not connected`). Mount Drive di Colab beda dgn S3/GCS --
rapuh thd akses bersamaan yg terlalu banyak. Sekarang `PREFETCH_WORKERS=1` (cuma 1 download
background sekaligus, bukan 4) -- tetap overlap dgn inferensi (thread beda), tapi TAK
menghantam Drive dgn banyak koneksi bersamaan.

**Retry + auto-remount:** disconnect Drive FUSE juga dikenal terjadi sporadis di Colab walau
TANPA paralelisme (flakiness platform, bukan cuma gara-gara load kita). Copy tile sekarang
dibungkus retry (sampai `MAX_RETRIES`x) -- kalau gagal krn Drive disconnect, coba `drive.mount`
ulang (`force_remount=True`) lalu retry, bukan langsung crash & buang progress satu tile.

**Pola dipakai (per tile):** 1 tile di-prefetch (background thread) ke `/content` -> infer
tile yg sudah siap -> mask (kecil, 1 band uint8) disalin balik ke **cache Drive** -> file
lokal dihapus segera -> prefetch tile berikutnya. Disk lokal cuma menampung 1-2 tile sekaligus.

**Resumable:** mask yang sudah ada di cache Drive (`MASK_CACHE_ROOT`) di-skip otomatis
(tak ikut di-download pula) -- kalau runtime Colab disconnect/timeout di tengah jalan,
jalankan ulang cell ini langsung lanjut dari tile yang belum selesai, BUKAN mengulang dari
awal. Tahun yang folder Drive-nya belum ada / kosong tetap di-skip dgn peringatan (bukan error).


In [ ]:
# === Bagian 3 -- Inferensi per tahun (prefetch aman + retry/remount + streaming + resumable) ===
import shutil
import time
from concurrent.futures import ThreadPoolExecutor

from tqdm.auto import tqdm

from forestwatch.inference.tile_inference import infer_tile

# PENTING: PREFETCH_WORKERS=1 (BUKAN >1) -- Drive FUSE mount Colab rapuh thd banyak koneksi
# bersamaan (sempat coba 4 -> "Transport endpoint is not connected", mount disconnect).
# 1 worker tetap kasih overlap I/O-vs-compute (thread beda dr main), tapi cuma 1 koneksi
# Drive aktif sekaligus -- aman.
PREFETCH_WORKERS = 1
PREFETCH_AHEAD = 1     # max tile "didownload duluan" yg ditahan di disk lokal sekaligus
MAX_RETRIES = 4         # retry copy kalau Drive FUSE disconnect (flakiness platform, bukan cuma load)
RETRY_BACKOFF_SEC = 5   # naik tiap retry (5, 10, 15, 20 detik)


def _copy_with_retry(src, dst):
    """copy2 dgn retry + auto-remount Drive kalau FUSE disconnect (errno 107/Transport)."""
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            shutil.copy2(src, dst)
            return
        except OSError as e:
            last_err = e
            print(f"  [retry {attempt}/{MAX_RETRIES}] copy gagal ({e}) -- coba remount Drive...")
            try:
                drive.mount("/content/drive", force_remount=True)
            except Exception as remount_err:  # noqa: BLE001
                print(f"  remount gagal: {remount_err}")
            time.sleep(RETRY_BACKOFF_SEC * attempt)
    raise last_err


mask_dirs = {}   # {year: Path Drive (cache) berisi mask_*.tif -- PERSISTEN, aman dr disconnect}
available_years = []

for year in YEARS:
    tile_dir = PROJECT_ROOT / TILE_DIR_OVERRIDE.get(year, TILE_DIR_TMPL.format(year=year))
    tile_glob = TILE_GLOB_OVERRIDE.get(year, TILE_GLOB_TMPL.format(year=year))
    tile_files = sorted(tile_dir.glob(tile_glob)) if tile_dir.exists() else []
    if not tile_files:
        print(f"[SKIP] {year}: folder '{tile_dir.name}' tak ada / kosong -- export blm selesai.")
        continue

    out_dir = MASK_CACHE_ROOT / f"masks_{year}"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Tile yg mask-nya SUDAH di cache -- skip total, tak ikut didownload pula (resumable).
    pending = [tp for tp in tile_files if not (out_dir / f"mask_{tp.name}").exists()]
    n_skipped = len(tile_files) - len(pending)
    n_done = 0
    print(f"[{year}] {len(tile_files)} tile ({n_skipped} sudah di cache, {len(pending)} perlu "
          f"diproses, prefetch={PREFETCH_WORKERS}x, batch={INFER_BATCH_SIZE}) -> {out_dir}")

    if pending:
        local_paths = {tp: LOCAL_TILE_STAGE_DIR / tp.name for tp in pending}
        with ThreadPoolExecutor(max_workers=PREFETCH_WORKERS) as pool:
            futures = {}

            def _submit(idx):
                if idx < len(pending):
                    tp = pending[idx]
                    futures[tp] = pool.submit(_copy_with_retry, tp, local_paths[tp])

            for i in range(min(PREFETCH_AHEAD, len(pending))):
                _submit(i)

            for idx, tile_path in enumerate(tqdm(pending, desc=f"Infer {year}")):
                local_tile = local_paths[tile_path]
                local_mask = LOCAL_TILE_STAGE_DIR / f"mask_{tile_path.name}"
                mask_path = out_dir / f"mask_{tile_path.name}"
                try:
                    futures[tile_path].result()  # tunggu download tile ini (sudah jalan di background)
                    _submit(idx + PREFETCH_AHEAD)  # mulai download tile berikutnya
                    infer_tile(
                        local_tile, local_mask, model,
                        device=DEVICE, patch_size=PATCH_SIZE, stride=STRIDE,
                        n_channels_image=IN_CHANNELS, tta=TTA, batch_size=INFER_BATCH_SIZE,
                    )
                    _copy_with_retry(local_mask, mask_path)  # mask kecil (1 band uint8 + lzw)
                    n_done += 1
                finally:
                    local_tile.unlink(missing_ok=True)
                    local_mask.unlink(missing_ok=True)  # disk lokal dibersihkan tiap iterasi

    mask_dirs[year] = out_dir
    available_years.append(year)
    print(f"[{year}] selesai: {n_done} baru diproses + {n_skipped} sudah ada (cache) "
          f"= {n_done + n_skipped}/{len(tile_files)} mask siap di {out_dir}")

print(f"\nTahun siap (ada mask): {available_years}")
assert T1_YEAR in available_years and T2_YEAR in available_years, (
    f"Pasangan resmi T1={T1_YEAR}/T2={T2_YEAR} butuh KEDUANYA tersedia utk deforestation.geojson."
)


## Bagian 4 -- Render landcover PNG (semua tahun tersedia) + deforestation.geojson

In [ ]:
# === Bagian 4a -- landcover_{tahun}.png + bounds (semua tahun yg ada mask) ===
from forestwatch.outputs.landcover_png import mosaic_masks_to_png

landcover_paths = {}
for year in available_years:
    mask_files = sorted(mask_dirs[year].glob("mask_*.tif"))
    png_path, bounds_path = mosaic_masks_to_png(
        mask_files,
        OUT_DIR / f"landcover_{year}.png",
        OUT_DIR / f"landcover_{year}_bounds.json",
    )
    landcover_paths[year] = (png_path, bounds_path)
    print(f"[{year}] {png_path.name} + {bounds_path.name}")


In [ ]:
# === Helper: prefix nama mask per tahun (konvensi nama tile TAK konsisten antar tahun) ===
# 2021 (T1_YEAR) & 2025 (T2_YEAR) dieksport lewat script lama (run_export.py) dgn token
# "t1"/"t2" di nama file (mis. mask_papua_t1_tile_00-....tif). 2022/2023/2024 dieksport lewat
# export_inference_years.ipynb yg pakai NAMA TAHUN LITERAL (mis. mask_papua_2023_tile_00-....tif).
YEAR_TILE_TOKEN = {T1_YEAR: "t1", T2_YEAR: "t2"}


def mask_prefix_for_year(year: int) -> str:
    token = YEAR_TILE_TOKEN.get(year, str(year))
    return f"mask_papua_{token}_tile_"


# === Bagian 4b -- deforestation.geojson (kumulatif T1->T2, kontrak WebGIS) ===
from forestwatch.inference.change_detection import detect_transitions

fc = detect_transitions(
    t1_dir=mask_dirs[T1_YEAR], t2_dir=mask_dirs[T2_YEAR],
    out_geojson=OUT_DIR / "deforestation.geojson",
    t1_prefix=mask_prefix_for_year(T1_YEAR), t2_prefix=mask_prefix_for_year(T2_YEAR),
    period_from=T1_YEAR, period_to=T2_YEAR, min_area_ha=MIN_AREA_HA,
)
print(f"deforestation.geojson: {len(fc['features'])} polygon perubahan ({T1_YEAR}->{T2_YEAR}).")

# --- Province join opsional (lihat config) ---
if PROVINCE_BOUNDARY_GEOJSON is not None:
    from shapely.geometry import shape
    from forestwatch.utils.io import load_json, save_geojson

    boundary_fc = load_json(PROVINCE_BOUNDARY_GEOJSON)
    boundary_polys = [
        (shape(f["geometry"]), f["properties"].get(PROVINCE_NAME_FIELD, "?"))
        for f in boundary_fc["features"]
    ]
    n_tagged = 0
    for feat in fc["features"]:
        centroid = shape(feat["geometry"]).centroid
        for poly, name in boundary_polys:
            if poly.contains(centroid):
                feat["properties"]["province"] = name
                n_tagged += 1
                break
    save_geojson(fc, OUT_DIR / "deforestation.geojson")
    print(f"Province join: {n_tagged}/{len(fc['features'])} polygon ter-tag provinsi.")
else:
    print("Province join DILEWATI (PROVINCE_BOUNDARY_GEOJSON=None) -- polygon tanpa tag provinsi.")


## Bagian 4c -- Tren tahunan (year-over-year, BUKAN kumulatif)

Beda dari Bagian 4b (yang cuma bandingkan T1 vs T2 langsung): ini bandingkan tahun
**berurutan** (2021->22, 22->23, dst.) supaya chart "Tren Perubahan Tahunan" di webgis bisa
pakai data asli, bukan estimasi/placeholder seperti sekarang.


In [ ]:
# === Bagian 4c -- tren tahunan (year-over-year, BUKAN kumulatif) ===
from forestwatch.outputs.statistics import summarize_geojson_transitions

sorted_years = sorted(available_years)
yearly_trend = []
cumulative_ha = 0.0

for i, year in enumerate(sorted_years):
    if i == 0:
        # Tahun pertama yang tersedia = baseline, tak ada "tahun sebelumnya" utk dibandingkan.
        yearly_trend.append({"year": year, "deforestation_ha": 0.0, "cumulative_ha": 0.0})
        continue

    prev_year = sorted_years[i - 1]
    pair_fc = detect_transitions(
        t1_dir=mask_dirs[prev_year], t2_dir=mask_dirs[year],
        out_geojson=OUT_DIR / f"deforestation_{prev_year}_{year}.geojson",
        t1_prefix=mask_prefix_for_year(prev_year), t2_prefix=mask_prefix_for_year(year),
        period_from=prev_year, period_to=year, min_area_ha=MIN_AREA_HA,
    )
    pair_summary = summarize_geojson_transitions(pair_fc)
    cumulative_ha += pair_summary["total_deforestation_ha"]
    yearly_trend.append({
        "year": year,
        "deforestation_ha": round(pair_summary["total_deforestation_ha"], 1),
        "cumulative_ha": round(cumulative_ha, 1),
    })
    print(f"[{prev_year}->{year}] {pair_summary['n_hotspots']} polygon perubahan, "
          f"{pair_summary['total_deforestation_ha']:.1f} ha")

print("\nTren tahunan (cumulative_ha):", yearly_trend)


## Bagian 5 -- statistics.json, legend.json, metrics.json, model_card.md

In [ ]:
# === Bagian 5 -- statistics.json + legend.json + metrics.json + model_card.md ===
from forestwatch.outputs.landcover_png import compute_per_class_area_ha_from_geotiff
from forestwatch.outputs.legend import build_legend_json
from forestwatch.outputs.model_card import render_model_card
from forestwatch.outputs.statistics import build_statistics_json, save_metrics_json
from forestwatch.utils.io import load_json

# Per-class area dari mask T2 (kondisi tutupan lahan terbaru)
from collections import defaultdict
per_class_area = defaultdict(float)
for f in sorted(mask_dirs[T2_YEAR].glob("mask_*.tif")):
    for name, ha in compute_per_class_area_ha_from_geotiff(f).items():
        per_class_area[name] += ha
per_class_area = {k: round(v, 1) for k, v in per_class_area.items()}

model_metrics = load_json(METRICS_SRC_JSON) if METRICS_SRC_JSON is not None else None
if model_metrics is None:
    print("PERINGATAN: METRICS_SRC_JSON=None -> model_metrics di statistics.json/metrics.json "
          "akan KOSONG (0.0), bukan dikarang. Isi config kalau file metrik training sudah ada.")

build_statistics_json(
    period_from=T1_YEAR, period_to=T2_YEAR,
    deforestation_geojson=fc, per_class_area_ha=per_class_area,
    model_metrics=model_metrics, out_path=OUT_DIR / "statistics.json",
    extra={"yearly_deforestation_ha": yearly_trend},  # tren year-over-year dari Bagian 4c
)
build_legend_json(out_path=OUT_DIR / "legend.json")
if model_metrics is not None:
    save_metrics_json(model_metrics, OUT_DIR / "metrics.json")
render_model_card(
    OUT_DIR / "model_card.md",
    n_parameters=n_params, metrics=model_metrics or {},
)
print("statistics.json, legend.json, model_card.md tersimpan ke", OUT_DIR)


## Bagian 6 -- Export ONNX dari model yang BARU dipakai inferensi

Diekspor langsung dari `model` di memori (bukan copy file `.onnx` lama) -- menjamin file ONNX
ini PERSIS sama dengan model yang menghasilkan mask/PNG/geojson di atas.


In [ ]:
# === Bagian 6 -- Export ONNX ke Output_Fix ===
from forestwatch.model.architecture import export_to_onnx

onnx_path = export_to_onnx(
    model, OUT_DIR / "model.onnx",
    in_channels=IN_CHANNELS, patch_size=PATCH_SIZE,
)
print("ONNX disimpan:", onnx_path)


## Bagian 7 -- Ringkasan & cross-check vs kontrak WebGIS

WebGIS (`webgis/backend/app/core/config.py`) butuh persis file-file ini di `Output_Fix`
(landcover per tahun bersifat tambahan/bonus -- backend saat ini baru baca 2021 & 2025).


In [ ]:
# === Bagian 7 -- Ringkasan isi Output_Fix ===
REQUIRED_BY_WEBGIS = [
    "landcover_2021.png", "landcover_2021_bounds.json",
    "landcover_2025.png", "landcover_2025_bounds.json",
    "deforestation.geojson", "statistics.json", "legend.json", "model.onnx",
]
present = {p.name for p in OUT_DIR.iterdir()}
print(f"Isi {OUT_DIR}:")
for name in sorted(present):
    print(" -", name)

missing = [f for f in REQUIRED_BY_WEBGIS if f not in present]
print("\nSemua file wajib kontrak WebGIS ADA." if not missing else f"\nMASIH KURANG: {missing}")

extra_landcover = sorted(y for y in available_years if y not in (T1_YEAR, T2_YEAR))
if extra_landcover:
    print(f"Bonus landcover tahun {extra_landcover} ikut tersimpan -- backend/frontend perlu "
          f"diperluas (VALID_YEARS di webgis/backend/app/core/config.py, "
          f"AVAILABLE_LANDCOVER_YEARS di frontend) kalau mau ditampilkan di slider tahun.")
if model_metrics is None:
    print("INGAT: metrics.json BELUM dibuat (METRICS_SRC_JSON=None) -- isi nanti & jalankan "
          "ulang Bagian 5 saja begitu file metrik training tersedia.")
if PROVINCE_BOUNDARY_GEOJSON is None:
    print("INGAT: per_province di statistics.json kosong utk data asli -- lihat catatan gap "
          "data batas provinsi di markdown paling atas.")
